In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader

import torch.optim as optim
import torchvision.transforms as transforms

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Data Loading

In [ ]:
df_train = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
df_test = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")

In [ ]:
y_train = df_train['label']
x_train = df_train.drop(columns=['label'])

In [ ]:
# pixels values are between 0 and 255
x_train = x_train/255.0
df_test = df_test/255.0

In [ ]:
x_train = x_train.values.reshape(-1, 1, 28, 28)  
df_test = df_test.values.reshape(-1, 1, 28, 28)

In [ ]:
plt.figure(figsize=(8,6))
for i in range(12):
    plt.subplot(3,4,i+1)
    plt.imshow(x_train[i][0],cmap='gray')
    plt.axis('off')
    plt.title(f"target: {y_train[i]}")

In [ ]:
torch.manual_seed(7)

In [ ]:
class MNISTDataset(Dataset):
    def __init__(self, images, labels=None):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.images[idx], self.labels[idx]
        else:
            return self.images[idx]

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0,5),(0.5))
])

batch_size = 64

from sklearn.model_selection import train_test_split

x_tr, x_val, y_tr, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

y_tr = y_tr.values
y_val = y_val.values

train_dataset = MNISTDataset(x_tr, y_tr)
val_dataset = MNISTDataset(x_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
class BaselineModel(nn.Module):
    def __init__(self, input_size, num_classes):
        super(BaselineModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 512)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, num_classes)
    
    def forward(self,x):
        x = x.view(-1, 28*28) # flatten iamge
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [ ]:
input_size = 28*28
num_classes = 10

model = BaselineModel(input_size, num_classes)
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)

## Train and evaluate Functions

In [ ]:
num_epochs = 10
batch_szie = 64

def train_model(model, train_loader, criterion, optimizer, num_epochs, scheduler=None):
    model.train()
    for epoch in range(num_epochs):
        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        if scheduler:
            scheduler.step()
            
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.6f}, LR: {optimizer.param_groups[0]['lr']:.8f}")



def evaluate_model(model, val_loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total


In [ ]:
train_model(model, train_loader, criterion, optimizer, num_epochs)

accuracy = evaluate_model(model, val_loader)
print(f'Baseline accuracy : {accuracy:.3f}%')

## SIMPLE CNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=5)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(32 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = nn.ReLU()(x)
        x = self.pool(x)
        x = self.dropout(x)
        x = x.view(-1, 32 * 12 * 12)
        x = self.fc1(x)
        x = nn.ReLU()(x)
        x = self.fc2(x)
        return x

In [ ]:
num_classes = 10
model = SimpleCNN(num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
num_epochs = 10
train_model(model, train_loader, criterion, optimizer, num_epochs)

accuracy_cnn = evaluate_model(model, val_loader)
print(f'Simple CNN accuracy : {accuracy_cnn:.4f}%')

In [ ]:
print("Hello")

## LARGE CNN

In [ ]:
class LargerCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(LargerCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 30, kernel_size=5)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(30, 15, kernel_size=3)
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(15 * 5 * 5, 128)
        self.fc2 = nn.Linear(128, 50)
        self.fc3 = nn.Linear(50, num_classes)

    def forward(self, x):
        x = self.pool(nn.ReLU()(self.conv1(x)))
        x = self.pool(nn.ReLU()(self.conv2(x)))
        x = self.dropout(x)
        x = x.view(-1, 15 * 5 * 5)
        x = nn.ReLU()(self.fc1(x))
        x = self.fc2(x)
        x = self.fc3(x)
        return x

In [ ]:
model = LargerCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 10
train_model(model, train_loader, criterion, optimizer, num_epochs)

accuracy_lcnn = evaluate_model(model, val_loader)
print(f'Large CNN accuracy : {accuracy_lcnn:.4f}%')

## Optimize large cnn

In [ ]:
class OptimCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(OptimCNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(64*7*7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(nn.ReLU()(self.fc1(x)))
        x = self.fc2(x)
        return x

In [ ]:
model = OptimCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [ ]:
num_epochs = 20
train_model(model, train_loader, criterion, optimizer, num_epochs, scheduler=scheduler)

accuracy_lcnn = evaluate_model(model, val_loader)
print(f'Improved CNN accuracy : {accuracy_lcnn:.4f}%')

## Predictions

In [ ]:
def predict(model, test_data, batch_size=64):
    model.eval()
    test_dataset = MNISTDataset(test_data)   # no labels
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    predictions = []
    with torch.no_grad():
        for images in test_loader:
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            predictions.extend(preds.cpu().numpy())
    return predictions


preds = predict(model, df_test)


submission = pd.DataFrame({
    "ImageId": np.arange(1, len(preds) + 1),
    "Label": preds
})
submission.to_csv("submission.csv", index=False)
print("submission.csv created!")


In [ ]:
pd.read_csv("/kaggle/working/submission.csv").info()